# Instacart SparkSQL — Refactored Notebook

Notebook này được refactor theo hướng rõ luồng xử lý, dễ chạy lại trên Google Colab và dễ giải thích trong báo cáo EDA.

Cấu trúc chính:
1. Cài đặt và khởi tạo môi trường Spark.
2. Tải dữ liệu Instacart từ Kaggle.
3. Đọc dữ liệu, chuẩn hóa kiểu dữ liệu và tạo TempViews.
4. Kiểm tra schema/số dòng.
5. Thực hiện các truy vấn Spark SQL và trực quan hóa EDA.

## 2. Tải các thư viện cần thiết

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os

from pathlib import Path
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")

print("Imports OK")

Imports OK


In [2]:
# Thiết lập cấu hình hiển thị đồng bộ cho toàn bộ notebook
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica', 'Calibri']
plt.rcParams['figure.titlesize'] = 16
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['grid.color'] = '#e2e2e2'
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.alpha'] = 0.5

COLOR_PRIMARY_PASTEL = "#8fa9c4"
COLOR_ACCENT_WARM = "#e6550d"
COLOR_ACCENT_COOL = "#2b8cbe"

## 4. Khởi tạo Spark Session


In [3]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Instacart_SQL") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


print("SparkSession khởi tạo thành công")
print(f"Version : {spark.version}")
print(f"App     : {spark.sparkContext.appName}")
print(f"Master  : {spark.sparkContext.master}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 06:37:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession khởi tạo thành công
Version : 4.1.1
App     : Instacart_SQL
Master  : local[*]


## 5. Đọc dữ liệu và tạo TempViews

Đọc 6 bảng CSV, chuẩn hóa tên/kiểu dữ liệu quan trọng, gộp bảng `prior` và `train`

In [ ]:
BASE_PATH = "hdfs://namenode:9000/instacart/raw"

def csv_path(file_name: str) -> str:
    return f"{BASE_PATH}/{file_name}"

csv_options = {
    "header": "true",
    "inferSchema": "true",
    "quote": '"',
    "escape": '"',
    "multiLine": "true",
}

orders_df = (
    spark.read.options(**csv_options)
    .csv(csv_path("orders.csv"))
    .withColumnRenamed("days_since_prior_order", "days_since_prior")
)

order_products_prior_df = spark.read.options(**csv_options).csv(csv_path("order_products__prior.csv"))

order_products_train_df = spark.read.options(**csv_options).csv(csv_path("order_products__train.csv"))

products_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("aisle_id", IntegerType(), True),
    StructField("department_id", IntegerType(), True),
])

products_df = (
    spark.read
    .options(header="true", quote='"', escape='"', multiLine="true")
    .schema(products_schema)
    .csv(csv_path("products.csv"))
)

aisles_df = spark.read.options(**csv_options).csv(csv_path("aisles.csv"))

departments_df = spark.read.options(**csv_options).csv(csv_path("departments.csv"))

orders_df = (
    orders_df
    .withColumn("order_id", F.col("order_id").cast(IntegerType()))
    .withColumn("user_id", F.col("user_id").cast(IntegerType()))
    .withColumn("eval_set", F.col("eval_set").cast(StringType()))
    .withColumn("order_number", F.col("order_number").cast(IntegerType()))
    .withColumn("order_dow", F.col("order_dow").cast(IntegerType()))
    .withColumn("order_hour_of_day", F.col("order_hour_of_day").cast(IntegerType()))
    .withColumn("days_since_prior", F.col("days_since_prior").cast(IntegerType()))
)

order_products_prior_df = (
    order_products_prior_df
    .withColumn("order_id", F.col("order_id").cast(IntegerType()))
    .withColumn("product_id", F.col("product_id").cast(IntegerType()))
    .withColumn("add_to_cart_order", F.col("add_to_cart_order").cast(IntegerType()))
    .withColumn("reordered", F.col("reordered").cast(IntegerType()))
)

order_products_train_df = (
    order_products_train_df
    .withColumn("order_id", F.col("order_id").cast(IntegerType()))
    .withColumn("product_id", F.col("product_id").cast(IntegerType()))
    .withColumn("add_to_cart_order", F.col("add_to_cart_order").cast(IntegerType()))
    .withColumn("reordered", F.col("reordered").cast(IntegerType()))
)

aisles_df = aisles_df.withColumn("aisle_id", F.col("aisle_id").cast(IntegerType()))

departments_df = departments_df.withColumn("department_id", F.col("department_id").cast(IntegerType()))

# Gộp prior và train bằng unionByName để an toàn hơn union theo vị trí cột.
order_products_all_df = order_products_prior_df.unionByName(order_products_train_df)

tables = {
    "orders": orders_df,
    "order_products_prior": order_products_prior_df,
    "order_products_train": order_products_train_df,
    "order_products_all": order_products_all_df,
    "products": products_df,
    "aisles": aisles_df,
    "departments": departments_df,
}

for table_name, table_df in tables.items():
    table_df.createOrReplaceTempView(table_name)
print("Đọc dữ liệu và đăng ký TempViews xong")

products_df.show(3, truncate=False)

products_df.printSchema()

## 6. Kiểm tra nhanh dữ liệu

Kiểm tra notebook đã đọc đủ bảng, đúng số cột và schema trước khi đi vào EDA.

In [ ]:
# Tạo danh sách rỗng để lưu thống kê tổng quan của từng bảng.
summary_rows = []

for table_name, table_df in tables.items():
    row_count = table_df.count()
    column_count = len(table_df.columns)
    summary_rows.append((table_name, row_count, column_count))

summary_pdf = pd.DataFrame(summary_rows, columns=["table_name", "rows", "columns"])

display(summary_pdf)

for table_name, table_df in tables.items():
    print(f"\nSchema: {table_name}")
    table_df.printSchema()

In [ ]:
orders_null_exprs = [
    F.sum(F.col(column_name).isNull().cast("int")).alias(column_name)
    for column_name in orders_df.columns
]

orders_nulls_df = orders_df.select(orders_null_exprs)

orders_nulls_df.show(truncate=False)

# 7. Spark SQL EDA

Mỗi phân tích bên dưới gồm 2 cell chính:
- **Cell Script**: chạy Spark SQL và tạo DataFrame kết quả.
- **Cell Chart**: chuyển kết quả nhỏ sang Pandas và vẽ biểu đồ.

## Query 1 — Tổng số đơn hàng theo thứ và giờ trong ngày

**Ý nghĩa:** Phân tích thời điểm khách hàng đặt hàng nhiều nhất để nhận diện khung giờ cao điểm.

**Kỹ thuật:** Truy vấn bảng `orders`, gom nhóm theo `order_dow` và `order_hour_of_day`, đếm số đơn hàng bằng `COUNT(*)`, đồng thời tính thêm `pct_of_total` bằng window function để biết mỗi ô ngày-giờ chiếm bao nhiêu phần trăm tổng đơn hàng.

In [ ]:
df_orders_by_day_hour = spark.sql("""
    SELECT
        order_dow,
        CASE order_dow
            WHEN 0 THEN 'Sunday'
            WHEN 1 THEN 'Monday'
            WHEN 2 THEN 'Tuesday'
            WHEN 3 THEN 'Wednesday'
            WHEN 4 THEN 'Thursday'
            WHEN 5 THEN 'Friday'
            WHEN 6 THEN 'Saturday'
        END AS day_name,
        order_hour_of_day,
        COUNT(*) AS total_orders,
        -- Tính tỷ trọng của từng ô ngày-giờ trên tổng số đơn hàng.
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
    FROM orders
    GROUP BY order_dow, order_hour_of_day
    ORDER BY order_dow, order_hour_of_day
""").cache()

pdf_orders_by_day_hour = df_orders_by_day_hour.toPandas()

display(pdf_orders_by_day_hour.head())

In [ ]:
day_order = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]

# Pivot dữ liệu để đưa về định dạng matrix cho heatmap
heatmap_data = pdf_orders_by_day_hour.pivot(index="day_name", columns="order_hour_of_day", values="total_orders")
heatmap_data = heatmap_data.reindex(day_order)
heatmap_data = heatmap_data.reindex(columns=range(24))

plt.figure(figsize=(15, 6))

ax = sns.heatmap(
    heatmap_data,
    cmap="crest",
    linewidths=0.05,
    linecolor="#f0f2f6",
    cbar_kws={'label': 'Tổng số đơn hàng'}
)

plt.title("Mật độ phân bổ đơn hàng theo khung giờ và thứ trong tuần", fontsize=15, fontweight="bold", pad=15)
plt.xlabel("Khung giờ trong ngày (0h - 23h)", labelpad=10)
plt.ylabel("Thứ trong tuần", labelpad=10)

plt.xticks(rotation=0)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

**Nhận xét:** Heatmap cho thấy nhu cầu đặt hàng tập trung rõ vào ban ngày, đặc biệt từ cuối buổi sáng đến chiều. Các ô màu đậm nằm nhiều ở Chủ Nhật và Thứ Hai, trong khi khung 0h–5h gần như là vùng thấp nhất. Điều này cho thấy hành vi đặt hàng của Instacart thiên về routine mua sắm đầu tuần/cuối tuần hơn là đặt rải đều trong ngày.

## Query 2 — Khoảng cách giữa hai lần đặt hàng

**Ý nghĩa:** Đo chu kỳ quay lại mua hàng của khách hàng thông qua số ngày giữa hai đơn liên tiếp. Biến này rất hữu ích khi phân tích retention, tần suất mua lại và phân khúc khách hàng.

**Kỹ thuật:** Sử dụng cột `days_since_prior` trong bảng `orders`; loại các dòng null vì đây là đơn đầu tiên của mỗi user, chưa có đơn trước đó để so sánh. Sau đó gom nhóm theo số ngày và đếm số đơn hàng ở từng mốc.

In [ ]:
df_days_prior = spark.sql("""
    SELECT
        days_since_prior,
        COUNT(*) AS total_orders
    FROM orders
    WHERE days_since_prior IS NOT NULL
    GROUP BY days_since_prior
    ORDER BY days_since_prior
""").cache()

pdf_days_prior = df_days_prior.toPandas()

display(pdf_days_prior)

In [ ]:
plt.figure(figsize=(15, 6))

colors = []
for x in pdf_days_prior["days_since_prior"]:
    if x in [7, 30]:
        colors.append(COLOR_ACCENT_WARM)
    else:
        colors.append(COLOR_PRIMARY_PASTEL)

ax = sns.barplot(
    data=pdf_days_prior,
    x="days_since_prior",
    y="total_orders",
    palette=colors,
    edgecolor="none"
)

for p in ax.patches:
    height = p.get_height()
    x_val = p.get_x() + p.get_width() / 2.
    # Chỉ hiển thị nhãn cho mốc 7 ngày và 30 ngày
    if int(p.get_x() + 0.5) in [7, 30]:
        ax.annotate(f'{height:,.0f}',
                    (x_val, height),
                    ha='center', va='bottom',
                    fontsize=10, fontweight='bold', color='#333333',
                    xytext=(0, 4), textcoords='offset points')

plt.title("Phân phối khoảng cách thời gian giữa hai lần đặt hàng liên tiếp", fontsize=15, fontweight="bold", pad=15)
plt.xlabel("Số ngày kể từ đơn hàng trước", labelpad=10)
plt.ylabel("Tổng số lượng đơn hàng", labelpad=10)

ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

**Nhận xét:** Phân phối có hai đỉnh rất rõ ở mốc **7 ngày** và **30 ngày**. Trong dataset này, mốc 7 ngày có **320,608 đơn**, còn mốc 30 ngày có **369,323 đơn**, cao nhất toàn bộ phân phối. Điều này phản ánh hai chu kỳ mua lại chính: mua hằng tuần và mua theo tháng, rất phù hợp để tạo nhóm khách hàng theo tần suất quay lại.

## Query 3 — Top 20 sản phẩm thường được thêm vào giỏ đầu tiên

**Ý nghĩa:** Tìm các sản phẩm mà hàng khách hàng thường chọn đầu tiên khi bắt đầu xây dựng giỏ hàng. Nhóm này thường là sản phẩm quen thuộc hoặc có nhu cầu mua lặp lại cao.

**Kỹ thuật:** Lọc `order_products_all` với `add_to_cart_order = 1`, join với `products` và `departments` để lấy tên sản phẩm và ngành hàng, sau đó đếm tần suất xuất hiện ở vị trí đầu tiên rồi lấy top 20.

In [ ]:
df_first_in_cart = spark.sql("""
    SELECT
        p.product_id,
        p.product_name,
        d.department,
        COUNT(*) AS first_in_cart_count
    FROM order_products_all op
    JOIN products p ON op.product_id = p.product_id
    JOIN departments d ON p.department_id = d.department_id
    WHERE op.add_to_cart_order = 1
    GROUP BY p.product_id, p.product_name, d.department
    ORDER BY first_in_cart_count DESC
    LIMIT 20
""").cache()

pdf_first_in_cart = df_first_in_cart.toPandas()

display(pdf_first_in_cart)

In [ ]:
pdf_first_plot = pdf_first_in_cart.sort_values("first_in_cart_count", ascending=True)

plt.figure(figsize=(12, 8))

# Sử dụng bảng màu xanh tươi mát phù hợp với đặc thù sản phẩm hữu cơ/tươi sống
ax = sns.barplot(
    data=pdf_first_plot,
    x="first_in_cart_count",
    y="product_name",
    hue="department",
    dodge=False,
    palette="GnBu_d"
)

# Hiển thị nhãn giá trị ở cuối mỗi thanh ngang để dễ theo dõi
for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.annotate(f'{width:,.0f}',
                    (width, p.get_y() + p.get_height() / 2.),
                    ha='left', va='center',
                    fontsize=9, color='#444444',
                    xytext=(5, 0), textcoords='offset points')

plt.title("Top 20 sản phẩm thường được đưa vào giỏ hàng đầu tiên", fontsize=15, fontweight="bold", pad=15)
plt.xlabel("Tần suất được thêm vào đầu tiên (lần)", labelpad=10)
plt.ylabel("")
plt.legend(title="Ngành hàng (Department)", loc="upper right", frameon=True, fontsize=12)

ax.xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

**Nhận xét:** Nhóm sản phẩm được thêm đầu tiên bị chi phối mạnh bởi các mặt hàng thiết yếu. `Banana` đứng đầu với **115,521 lần**, tiếp theo là `Bag of Organic Bananas` với **82,877 lần**; sau đó mới đến `Organic Whole Milk`, `Organic Strawberries` và các sản phẩm produce, dairy khác. Điều này cho thấy các nhóm sản phẩm thường là sản phẩm đầu tiên để khách hàng bắt đầu giỏ hàng.

## Query 4 — Phân phối kích thước giỏ hàng

**Ý nghĩa:** Xác định một đơn hàng thường có bao nhiêu sản phẩm, từ đó hiểu quy mô basket phổ biến và phát hiện phần đuôi dài của các đơn hàng rất lớn.

**Kỹ thuật:** Tính `basket_size` bằng `COUNT(product_id)` trên từng `order_id` trong `order_products_all`.

In [ ]:
df_basket_size = spark.sql("""
    WITH basket_sizes AS (
        SELECT
            order_id,
            COUNT(product_id) AS basket_size
        FROM order_products_all
        GROUP BY order_id
    )
    SELECT
        basket_size,
        COUNT(*) AS total_orders
    FROM basket_sizes
    GROUP BY basket_size
    ORDER BY basket_size
""").cache()

pdf_basket_size = df_basket_size.toPandas()

display(pdf_basket_size.head())


In [ ]:
# Lọc giỏ hàng nhỏ hơn hoặc bằng 35 món để biểu đồ tập trung
pdf_basket_plot = pdf_basket_size[pdf_basket_size["basket_size"] <= 35].copy()

plt.figure(figsize=(15, 6))

# Tạo bảng màu chuyển sắc (gradient) tinh tế dựa trên quy mô giỏ hàng
colors = sns.color_palette("ch:start=.2,rot=-.3_r", len(pdf_basket_plot))

ax = sns.barplot(
    data=pdf_basket_plot,
    x="basket_size",
    y="total_orders",
    palette=colors,
    edgecolor="none"
)

plt.title("Phân phối quy mô giỏ hàng", fontsize=15, fontweight="bold", pad=15)
plt.xlabel("Số lượng sản phẩm trong giỏ hàng", labelpad=10)
plt.ylabel("Tổng số đơn hàng", labelpad=10)
plt.legend(loc="upper right", frameon=True)

ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

**Nhận xét:** Phân phối basket size giúp xác định quy mô đơn hàng phổ biến. Nếu phần lớn đơn nằm quanh khoảng 4–10 món, có thể xem đây là hành vi mua hàng thông thường

## Query 5 — Customer Journey theo số lần đặt hàng

**Ý nghĩa:** Theo dõi hành vi thay đổi theo từng lần mua của khách hàng: giỏ hàng trung bình có ổn định không và tỷ lệ sản phẩm mua lại tăng nhanh ở giai đoạn nào.

**Kỹ thuật:** Join `orders` với `order_products_all`, tính số món và số món reordered ở từng đơn hàng, sau đó gom theo `order_number`.

In [ ]:
df_customer_journey = spark.sql("""
    WITH order_metrics AS (
        SELECT
            o.order_id,
            o.order_number,
            COUNT(op.product_id) AS basket_size,
            SUM(op.reordered) AS total_reordered
        FROM orders o
        JOIN order_products_all op ON o.order_id = op.order_id
        GROUP BY o.order_id, o.order_number
    )
    SELECT
        order_number,
        COUNT(order_id) AS total_orders,
        ROUND(AVG(basket_size), 2) AS avg_basket_size,
        -- Weighted reorder rate: tổng reordered / tổng sản phẩm ở cùng order_number.
        ROUND(SUM(total_reordered) * 100.0 / SUM(basket_size), 2) AS reorder_rate
    FROM order_metrics
    WHERE order_number <= 50
    GROUP BY order_number
    ORDER BY order_number
""").cache()

pdf_customer_journey = df_customer_journey.toPandas()

display(pdf_customer_journey.head())


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

# Đường biểu diễn Quy mô giỏ hàng trung bình (Trục Y bên trái)
color_basket = "#2b5c8f"
line1 = ax1.plot(
    pdf_customer_journey["order_number"],
    pdf_customer_journey["avg_basket_size"],
    marker="o",
    color=color_basket,
    linewidth=2,
    markersize=6,
    label="Quy mô giỏ trung bình"
)
ax1.set_xlabel("Thứ tự đơn hàng của khách hàng (Order Number)", fontweight="bold", labelpad=10)
ax1.set_ylabel("")
ax1.tick_params(axis='y', labelcolor=color_basket)
ax1.yaxis.tick_left()
ax1.yaxis.set_label_position("left")
ax1.grid(True, linestyle="--", alpha=0.3)

# Đường biểu diễn Tỷ lệ mua lại weighted/item-level (Trục Y bên phải)
ax2 = ax1.twinx()
color_reorder = COLOR_ACCENT_WARM
line2 = ax2.plot(
    pdf_customer_journey["order_number"],
    pdf_customer_journey["reorder_rate"],
    marker="s",
    color=color_reorder,
    linewidth=2,
    markersize=6,
    label="Tỷ lệ mua lại weighted (%)"
)
ax2.set_ylabel("")
ax2.tick_params(axis='y', labelcolor=color_reorder)
ax2.yaxis.tick_right()
ax2.yaxis.set_label_position("right")
ax2.grid(False)

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="lower right", frameon=True)

plt.title("Sự thay đổi hành vi theo tần suất mua sắm", fontsize=15, fontweight="bold", pad=15)

sns.despine(ax=ax1, left=False, right=True, top=True, bottom=False)
sns.despine(ax=ax2, left=True, right=False, top=True, bottom=False)

fig.tight_layout()
plt.show()


**Nhận xét:** Ở đơn đầu tiên, reorder rate bằng **0%** vì chưa có lịch sử mua trước đó. Từ đơn thứ 2 trở đi, tỷ lệ mua lại tăng nhanh; kết quả trên dataset này thường cho thấy đơn thứ 2 khoảng **28.84%**, đơn thứ 3 khoảng **40.51%**, và đến đơn thứ 5 đã vượt **52%**. Trong khi đó, basket size trung bình dao động quanh khoảng 10 món, nghĩa là sự thay đổi lớn nhất theo journey nằm ở mức độ lặp lại sản phẩm quen thuộc hơn là số lượng món trong mỗi giỏ.

## Query 6 — So sánh sản phẩm Organic và Non-Organic

**Ý nghĩa:** So sánh quy mô bán ra và mức độ mua lại giữa nhóm sản phẩm có yếu tố “organic” và nhóm còn lại, từ đó đánh giá liệu organic có tạo loyalty tốt hơn hay không.

**Kỹ thuật:** Tạo nhãn nhanh bằng rule `LOWER(product_name) LIKE '%organic%'`. Sau đó tính `total_sold` bằng `COUNT(product_id)` và `reorder_rate` bằng tỷ lệ `SUM(reordered) / COUNT(product_id)`. Đây là heuristic dựa trên tên sản phẩm, không phải nhãn chính thức của dataset.

In [ ]:
df_organic = spark.sql("""
    WITH product_flags AS (
        SELECT
            op.product_id,
            op.reordered,
            CASE
                WHEN LOWER(p.product_name) LIKE '%organic%' THEN 'Organic'
                ELSE 'Non-Organic'
            END AS product_category
        FROM order_products_all op
        JOIN products p ON op.product_id = p.product_id
    )
    SELECT
        product_category,
        COUNT(product_id) AS total_sold,
        ROUND(SUM(reordered) * 100.0 / COUNT(product_id), 2) AS reorder_rate
    FROM product_flags
    GROUP BY product_category
    ORDER BY product_category
""").cache()

pdf_organic = df_organic.toPandas()

display(pdf_organic)

In [ ]:
pdf_organic_plot = pdf_organic.copy()
pdf_organic_plot["product_category"] = pd.Categorical(
    pdf_organic_plot["product_category"],
    categories=["Organic", "Non-Organic"],
    ordered=True,
)
pdf_organic_plot = pdf_organic_plot.sort_values("product_category")

# Thiết lập lát cắt tách nhẹ (explode) cho Organic để nhấn mạnh
explode_values = [0.08, 0]
colors_pie = ["#7fcdbb", COLOR_PRIMARY_PASTEL]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. Biểu đồ tròn - Tỷ trọng sản lượng
wedges, texts, autotexts = axes[0].pie(
    pdf_organic_plot["total_sold"],
    labels=pdf_organic_plot["product_category"],
    autopct="%1.1f%%",
    startangle=90,
    colors=colors_pie,
    explode=explode_values,
    shadow=False,
    textprops=dict(color="#333333")
)
# Làm nhạt màu phần trăm hiển thị bên trong lát bánh cho dễ nhìn
plt.setp(autotexts, size=10, weight="bold")
axes[0].set_title("Tỷ trọng sản lượng bán ra", fontsize=13, fontweight="bold", pad=10)

# 2. Biểu đồ cột - Tỷ lệ mua lại (Reorder Rate)
sns.barplot(
    data=pdf_organic_plot,
    x="product_category",
    y="reorder_rate",
    palette=colors_pie,
    ax=axes[1],
    edgecolor="none"
)
for index, value in enumerate(pdf_organic_plot["reorder_rate"]):
    axes[1].text(index, value / 2, f"{value:.1f}%", ha="center", va="center", color="#333333", fontweight="bold", fontsize=12)

axes[1].set_title("Tỷ lệ mua lại - Reorder Rate", fontsize=13, fontweight="bold", pad=10)
axes[1].set_ylabel("")
axes[1].set_xlabel("")

sns.despine(ax=axes[1], left=True, bottom=True)
plt.suptitle("Phân tích hành vi tiêu dùng: Sản phẩm Organic vs Non-Organic", fontsize=15, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

**Nhận xét:** Nhóm Non-Organic vẫn chiếm sản lượng lớn hơn nhiều với **23,162,774 lượt mua**, trong khi Organic có **10,656,332 lượt mua**. Tuy nhiên, Organic có reorder rate cao hơn (**63.54%** so với **56.92%**), cho thấy nhóm này tuy nhỏ hơn về volume nhưng có mức độ trung thành tốt hơn.

## Query 7 — Bảng pair dùng chung và Top 15 cặp sản phẩm thường mua chung

**Ý nghĩa:** Phát hiện các cặp sản phẩm có xu hướng xuất hiện chung trong cùng giỏ hàng để phục vụ phân tích bundle, cross-sell và recommendation.

**Kỹ thuật:** Tạo bảng nền `product_pair_base` bằng self-join `order_products_all` theo cùng `order_id`, dùng điều kiện `a.product_id < b.product_id` để tránh đếm trùng cặp A-B và B-A. Trước khi self-join, chỉ giữ các sản phẩm ứng viên gồm nhóm volume cao, Blockbuster và Hidden Gem để giảm tải xử lý. Bảng này được dùng lại cho Top pair và Query Hidden Gem × Blockbuster.

In [ ]:
df_product_pair_base = spark.sql("""
    WITH product_metrics AS (
        SELECT
            op.product_id,
            COUNT(*) AS total_purchases,
            AVG(op.reordered) AS reorder_rate
        FROM order_products_all op
        GROUP BY op.product_id
    ),
    product_ranked AS (
        SELECT
            pm.product_id,
            p.product_name,
            pm.total_purchases,
            pm.reorder_rate,
            CASE WHEN LOWER(p.product_name) LIKE '%organic%' THEN 1 ELSE 0 END AS is_organic,
            ROW_NUMBER() OVER (ORDER BY pm.total_purchases DESC) AS purchase_rank
        FROM product_metrics pm
        JOIN products p ON pm.product_id = p.product_id
    ),
    product_segments AS (
        SELECT
            product_id,
            product_name,
            total_purchases,
            reorder_rate,
            is_organic,
            CASE
                WHEN purchase_rank <= 50 THEN 'Blockbuster'
                WHEN total_purchases BETWEEN 100 AND 1000 AND reorder_rate > 0.85 THEN 'Hidden Gem'
                WHEN total_purchases > 50000 THEN 'High Volume'
                ELSE 'Other'
            END AS product_segment
        FROM product_ranked
    ),
    candidate_products AS (
        SELECT
            product_id
        FROM product_segments
        WHERE total_purchases > 50000
           OR product_segment IN ('Hidden Gem', 'Blockbuster')
    ),
    pair_lines AS (
        SELECT
            a.order_id,
            a.product_id AS product_a_id,
            b.product_id AS product_b_id
        FROM order_products_all a
        JOIN candidate_products ca ON a.product_id = ca.product_id
        JOIN order_products_all b
            ON a.order_id = b.order_id
            AND a.product_id < b.product_id
        JOIN candidate_products cb ON b.product_id = cb.product_id
    )
    SELECT
        pa.product_id AS product_a_id,
        pb.product_id AS product_b_id,
        pa.product_name AS product_a,
        pb.product_name AS product_b,
        pa.is_organic AS is_organic_a,
        pb.is_organic AS is_organic_b,
        pa.product_segment AS segment_a,
        pb.product_segment AS segment_b,
        pa.total_purchases AS total_purchases_a,
        pb.total_purchases AS total_purchases_b,
        ROUND(pa.reorder_rate * 100.0, 2) AS reorder_rate_a,
        ROUND(pb.reorder_rate * 100.0, 2) AS reorder_rate_b,
        -- Số lần hai sản phẩm xuất hiện chung trong cùng đơn.
        COUNT(*) AS co_occurrence
    FROM pair_lines pl
    JOIN product_segments pa ON pl.product_a_id = pa.product_id
    JOIN product_segments pb ON pl.product_b_id = pb.product_id
    WHERE (pa.total_purchases > 50000 AND pb.total_purchases > 50000)
       OR (pa.product_segment = 'Hidden Gem' AND pb.product_segment = 'Blockbuster')
       OR (pa.product_segment = 'Blockbuster' AND pb.product_segment = 'Hidden Gem')
    GROUP BY
        pa.product_id,
        pb.product_id,
        pa.product_name,
        pb.product_name,
        pa.is_organic,
        pb.is_organic,
        pa.product_segment,
        pb.product_segment,
        pa.total_purchases,
        pb.total_purchases,
        pa.reorder_rate,
        pb.reorder_rate
""").cache()

df_product_pair_base.createOrReplaceTempView("product_pair_base")

# Từ bảng dùng chung, lấy top 15 cặp sản phẩm volume cao thường được mua chung.
df_product_pairs = spark.sql("""
    SELECT
        product_a,
        product_b,
        is_organic_a,
        is_organic_b,
        segment_a,
        segment_b,
        -- Số lần xuất hiện chung.
        co_occurrence
    FROM product_pair_base
    -- Top product pair chỉ xét hai sản phẩm volume cao để tránh nhiễu từ sản phẩm quá ngách.
    WHERE total_purchases_a > 50000
      AND total_purchases_b > 50000
    ORDER BY co_occurrence DESC
    LIMIT 15
""").cache()

pdf_product_pairs = df_product_pairs.toPandas()

display(pdf_product_pairs)


In [ ]:
pdf_pairs_plot = pdf_product_pairs.copy()
pdf_pairs_plot["pair_label"] = pdf_pairs_plot["product_a"].str[:22] + " + " + pdf_pairs_plot["product_b"].str[:22]
pdf_pairs_top15 = pdf_pairs_plot.sort_values("co_occurrence", ascending=True).tail(15).copy()

pdf_pairs_top15["pair_type"] = np.select(
    [
        (pdf_pairs_top15["is_organic_a"] == 1) & (pdf_pairs_top15["is_organic_b"] == 1),
        (pdf_pairs_top15["is_organic_a"] == 1) | (pdf_pairs_top15["is_organic_b"] == 1),
    ],
    ["Organic + Organic", "Hỗn hợp"],
    default="Non-Organic + Non-Organic",
)

# Ánh xạ màu sắc sang các nhóm cặp sản phẩm
color_map = {
    "Organic + Organic": "#7fcdbb",
    "Hỗn hợp": "#a6bddb",
    "Non-Organic + Non-Organic": COLOR_PRIMARY_PASTEL
}

plt.figure(figsize=(14, 8))
ax = sns.barplot(
    data=pdf_pairs_top15,
    x="co_occurrence",
    y="pair_label",
    hue="pair_type",
    palette=color_map,
    dodge=False,
    edgecolor="none"
)

plt.title("Top 15 cặp sản phẩm được mua cùng nhau nhiều nhất", fontsize=15, fontweight="bold", pad=15)
plt.xlabel("Số lần xuất hiện đồng thời trong giỏ hàng", labelpad=10)
plt.ylabel("")
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))

plt.legend(title="Tính chất cặp sản phẩm", loc="upper right", frameon=True, fontsize=10)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

**Nhận xét:** Top pair tập trung rất mạnh vào các sản phẩm thiết yếu thuộc produce. Hai cặp đứng đầu trong output hiện tại là `Bag of Organic Bananas + Organic Hass Avocado` (**64,761 lần**) và `Bag of Organic Bananas + Organic Strawberries` (**64,702 lần**). Việc nhiều cặp top đều có Banana/Organic Banana cho thấy đây là sản phẩm trung tâm trong basket routine và có giá trị cao cho gợi ý mua kèm.

## Query 8 — Sản lượng bán và tỷ lệ mua lại của các aisle thuộc ngành hàng Produce

**Ý nghĩa:** Đi sâu vào department `produce` để biết aisle nào đang đóng góp sản lượng lớn và aisle nào có mức độ mua lại cao. Đây là ngành hàng quan trọng vì xuất hiện dày đặc trong các top product và product-pair.

**Kỹ thuật:** Join `order_products_all` → `products` → `aisles` → `departments`, lọc `department = 'produce'`, sau đó gom nhóm theo `aisle`. `total_sales` đo tổng lượt mua sản phẩm, còn `reorder_rate` đo tỷ lệ sản phẩm được mua lại trong từng aisle.

In [ ]:
df_produce_aisles = spark.sql("""
    SELECT
        a.aisle,
        COUNT(op.product_id) AS total_sales,
        ROUND(AVG(op.reordered) * 100, 2) AS reorder_rate
    FROM order_products_all op
    JOIN products p ON op.product_id = p.product_id
    JOIN aisles a ON p.aisle_id = a.aisle_id
    JOIN departments d ON p.department_id = d.department_id
    WHERE d.department = 'produce'
    GROUP BY a.aisle
    ORDER BY total_sales DESC
""").cache()

pdf_produce_aisles = df_produce_aisles.toPandas()

display(pdf_produce_aisles.head())

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 7))

# Vẽ biểu đồ cột (Sản lượng bán ra - Trực Y bên trái)
ax1.bar(
    pdf_produce_aisles["aisle"].str.title(),
    pdf_produce_aisles["total_sales"],
    color=color_sales,
    alpha=0.85,
    label="Tổng sản lượng bán ra"
)
ax1.set_xlabel("Các nhóm sản phẩm trong ngành Produce", fontweight="bold", labelpad=12)
ax1.set_ylabel("")
ax1.tick_params(axis='y', labelcolor=color_basket)
ax1.yaxis.tick_left() # Ép hiển thị số ở bên TRÁI
ax1.yaxis.set_label_position("left")
ax1.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
ax1.grid(True, linestyle="--", alpha=0.3)

# Vẽ biểu đồ đường (Tỷ lệ mua lại - Trục Y bên phải)
ax2 = ax1.twinx()
ax2.plot(
    pdf_produce_aisles["aisle"].str.title(),
    pdf_produce_aisles["reorder_rate"],
    marker="s",
    color=color_reorder,
    linewidth=2.5,
    markersize=8,
    label="Tỷ lệ mua lại (%)"
)
ax2.set_ylabel("")
ax2.tick_params(axis='y', labelcolor=color_reorder)
ax2.yaxis.tick_right() # Ép hiển thị số ở bên PHẢI
ax2.yaxis.set_label_position("right")
ax2.grid(False)

# Gộp chú thích
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", frameon=True)

plt.title("Phân tích ngành hàng Produce: Sản lượng bán ra & Tỷ lệ mua lại", fontsize=15, fontweight="bold", pad=15)

# --- SỬA LẠI ĐỂ TÁCH BIỆT TRỤC SỐ SANG HAI BÊN ---
sns.despine(ax=ax1, left=False, right=True, top=True, bottom=False)
sns.despine(ax=ax2, left=True, right=False, top=True, bottom=False)

fig.tight_layout()
plt.show()

**Nhận xét:** Trong department Produce, các aisle như trái cây tươi, rau củ tươi và nhóm packaged produce thường là những nhóm đóng góp sản lượng lớn nhất. Khi kết hợp cột sản lượng và đường reorder rate, có thể tách hai ý: aisle có volume cao là nhóm kéo traffic chính, còn aisle có reorder rate cao là nhóm có tính lặp lại tốt. Vì vậy, không nên chỉ nhìn tổng sales; một aisle nhỏ hơn nhưng reorder rate cao vẫn có thể quan trọng cho retention.

## Query 9 — Department Matrix: độ phủ và lòng trung thành

**Ý nghĩa:** So sánh các department theo 3 chiều: số khách hàng tiếp cận, tỷ lệ mua lại và số món trung bình mỗi đơn. Ma trận này giúp phân biệt nhóm ngành hàng vừa có reach cao vừa có loyalty cao với nhóm có reach hoặc loyalty thấp.

**Kỹ thuật:** Join 4 bảng `orders`, `order_products_all`, `products`, `departments`; tính `total_unique_buyers` bằng `COUNT(DISTINCT user_id)`, `reorder_rate` bằng `SUM(reordered) / COUNT(*)`, và `avg_items_per_order` bằng tổng dòng sản phẩm chia cho số đơn khác nhau. Bubble size biểu diễn `avg_items_per_order`.

In [ ]:
df_department_matrix = spark.sql("""
    SELECT
        d.department,
        COUNT(DISTINCT o.user_id) AS total_unique_buyers,
        ROUND(SUM(op.reordered) * 100.0 / COUNT(*), 2) AS reorder_rate,
        ROUND(COUNT(op.product_id) * 1.0 / COUNT(DISTINCT o.order_id), 2) AS avg_items_per_order
    FROM orders o
    JOIN order_products_all op ON o.order_id = op.order_id
    JOIN products p ON op.product_id = p.product_id
    JOIN departments d ON p.department_id = d.department_id
    GROUP BY d.department
""").cache()

pdf_department_matrix = df_department_matrix.toPandas()
pdf_department_matrix["reorder_rate"] = pdf_department_matrix["reorder_rate"].astype(float)
pdf_department_matrix["avg_items_per_order"] = pdf_department_matrix["avg_items_per_order"].astype(float)

display(pdf_department_matrix)

In [ ]:
# Tính toán các giá trị trung vị phục vụ phân chia ranh giới 4 vùng chiến lược
reach_median = pdf_department_matrix["total_unique_buyers"].median()
loyalty_median = pdf_department_matrix["reorder_rate"].median()

plt.figure(figsize=(14, 9))

# Vẽ đồ thị bong bóng với kích thước đại diện cho mức đóng góp vào giỏ hàng (avg_items)
sns.scatterplot(
    data=pdf_department_matrix,
    x="total_unique_buyers",
    y="reorder_rate",
    size="avg_items_per_order",
    sizes=(100, 2000),
    hue="reorder_rate",
    palette="viridis",
    alpha=0.8,
    legend=False,
)

for _, row in pdf_department_matrix.iterrows():
    plt.text(
        row["total_unique_buyers"],
        row["reorder_rate"] + 0.6,
        row["department"].title(),
        ha="center",
        fontsize=9,
        fontweight="bold",
        color="#333333"
    )

plt.axvline(x=reach_median, color="#cccccc", linestyle="--", linewidth=1.2)
plt.axhline(y=loyalty_median, color="#cccccc", linestyle="--", linewidth=1.2)
plt.gca().xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

plt.title("Ma trận Ngành hàng: Mức độ tiếp cận khách hàng vs Tỷ lệ đặt mua lại", fontsize=16, fontweight="bold", pad=15)
plt.xlabel("Tổng số khách hàng tiếp cận", labelpad=10)
plt.ylabel("Tỷ lệ đặt mua lại (%)", labelpad=10)
plt.tight_layout()
plt.show()

**Nhận xét:** `produce` và `dairy eggs` là hai department mạnh nhất về độ phủ: lần lượt khoảng **194,331** và **191,861** khách hàng duy nhất. `dairy eggs` có reorder rate cao nhất trong nhóm lớn (**67.02%**), còn `produce` có basket intensity nổi bật với **3.95 món/đơn**. Ngược lại, `personal care`, `pantry` và `international` có reorder rate thấp hơn, thể hiện đặc tính mua không đều hoặc ít lặp lại hơn.

## Query 10 — Danh mục sản phẩm đầu tiên và số lượng đơn hàng trọn đời của user

**Ý nghĩa:** Kiểm tra liệu department của sản phẩm đầu tiên trong đơn đầu tiên có liên quan đến số đơn hàng trọn đời trung bình của user hay không.

**Kỹ thuật:** Lọc đơn đầu tiên của mỗi user (`order_number = 1`) và sản phẩm được thêm đầu tiên (`add_to_cart_order = 1`), lấy department của sản phẩm đó, sau đó join với `MAX(order_number)` để tính `lifetime_orders`. Chỉ giữ các department có hơn 1.000 user để tránh nhóm quá nhỏ gây nhiễu.

In [ ]:
df_first_order_clv = spark.sql("""
    WITH user_first_order_item AS (
        -- Lấy sản phẩm đầu tiên được thêm vào giỏ trong đơn hàng đầu tiên (order_number = 1) của mỗi user
        SELECT
            o.user_id,
            op.product_id
        FROM orders o
        JOIN order_products_all op ON o.order_id = op.order_id
        WHERE o.order_number = 1 AND op.add_to_cart_order = 1
    ),
    user_lifetime_orders AS (
        -- Lấy tổng số đơn hàng trọn đời của mỗi user (MAX của order_number)
        SELECT
            user_id,
            MAX(order_number) AS lifetime_orders
        FROM orders
        GROUP BY user_id
    ),
    user_profile AS (
        SELECT
            ufoi.user_id,
            p.department_id,
            ulo.lifetime_orders
        FROM user_first_order_item ufoi
        JOIN products p ON ufoi.product_id = p.product_id
        JOIN user_lifetime_orders ulo ON ufoi.user_id = ulo.user_id
    )
    SELECT
        d.department,
        COUNT(DISTINCT up.user_id) AS total_users,
        ROUND(AVG(up.lifetime_orders), 2) AS avg_lifetime_orders
    FROM user_profile up
    JOIN departments d ON up.department_id = d.department_id
    GROUP BY d.department
    HAVING COUNT(DISTINCT up.user_id) > 1000
    ORDER BY avg_lifetime_orders DESC
""").cache()

pdf_first_order_clv = df_first_order_clv.toPandas()

display(pdf_first_order_clv.head())

In [ ]:
plt.figure(figsize=(14, 7))
colors = [COLOR_ACCENT_WARM if x == pdf_first_order_clv["avg_lifetime_orders"].max() else COLOR_PRIMARY_PASTEL for x in pdf_first_order_clv["avg_lifetime_orders"]]

ax = sns.barplot(
    data=pdf_first_order_clv,
    x="avg_lifetime_orders",
    y="department",
    palette=colors,
    edgecolor="none"
)

for container in ax.containers:
    ax.bar_label(container, fmt=" %.1f đơn", padding=3, color="#333333", fontweight="bold")

plt.title("Sản phẩm đơn đầu tiên dự đoán số lượng đơn hàng trọn đời (CLV) của khách hàng", fontsize=15, fontweight="bold", pad=15)
plt.xlabel("Số lượng đơn hàng trọn đời trung bình (đơn/khách hàng)", labelpad=10)
plt.ylabel("")
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

**Nhận xét:** Nhóm user bắt đầu bằng sản phẩm thuộc `babies` có `avg_lifetime_orders` cao nhất trong lần chạy trên dataset này (**18.24 đơn/user**), nhưng quy mô mẫu chỉ hơn 1,000 user nên cần diễn giải thận trọng. Các department lớn hơn như `dairy eggs` (**17.64**) và `produce` (**17.10**) đáng tin cậy hơn vì có số user lớn. Kết quả này nên hiểu là tín hiệu liên quan đến retention, không phải quan hệ nhân quả trực tiếp giữa sản phẩm đầu tiên và CLV.

## Query 11 — Cặp Hidden Gem và Blockbuster được mua chung nhiều nhất

**Ý nghĩa:** Xem các sản phẩm ngách nhưng có tỷ lệ mua lại cao có thường đi cùng các sản phẩm chủ lực hay không. Đây là hướng phân tích hữu ích cho gợi ý mua kèm: dùng sản phẩm phổ biến để kéo sự chú ý đến sản phẩm ngách có loyalty tốt.

**Kỹ thuật:** Tái sử dụng `product_pair_base` từ Query 7, lọc đúng hai hướng `Hidden Gem × Blockbuster` và `Blockbuster × Hidden Gem`. Dùng `CASE WHEN` để chuẩn hóa tên cột thành `hidden_gem` và `blockbuster`, sau đó sắp xếp theo `co_occurrence` giảm dần.

In [ ]:
df_hidden_blockbuster_pairs = spark.sql("""
    SELECT
        CASE
            WHEN segment_a = 'Hidden Gem' THEN product_a
            ELSE product_b
        END AS hidden_gem,
        CASE
            WHEN segment_a = 'Blockbuster' THEN product_a
            ELSE product_b
        END AS blockbuster,
        -- Số lần hai sản phẩm xuất hiện chung trong cùng đơn hàng.
        co_occurrence,
        CASE
            WHEN segment_a = 'Hidden Gem' THEN reorder_rate_a
            ELSE reorder_rate_b
        END AS hidden_gem_reorder_rate,
        CASE
            WHEN segment_a = 'Blockbuster' THEN total_purchases_a
            ELSE total_purchases_b
        END AS blockbuster_total_purchases
    FROM product_pair_base
    WHERE (segment_a = 'Hidden Gem' AND segment_b = 'Blockbuster')
       OR (segment_a = 'Blockbuster' AND segment_b = 'Hidden Gem')
    ORDER BY co_occurrence DESC
    LIMIT 20
""").cache()

pdf_hidden_blockbuster_pairs = df_hidden_blockbuster_pairs.toPandas()

display(pdf_hidden_blockbuster_pairs)


In [ ]:
pdf_hidden_blockbuster_graph = pdf_hidden_blockbuster_pairs.copy()

# Khởi tạo đồ thị không hướng từ pandas edgelist
graph = nx.from_pandas_edgelist(
    pdf_hidden_blockbuster_graph,
    source="hidden_gem",
    target="blockbuster",
    edge_attr=["co_occurrence"],
    create_using=nx.Graph(),
)

# Định vị màu sắc cho từng nhóm node để có độ tương phản thẩm mỹ cao
hidden_gem_nodes = set(pdf_hidden_blockbuster_graph["hidden_gem"])
node_colors = ["#f768a1" if node in hidden_gem_nodes else "#7fcdbb" for node in graph.nodes()]

# Tính toán kích cỡ node dựa trên tổng trọng số liên kết (mức độ mua kèm)
node_sizes = [
    800 + 35 * sum(edge_data["co_occurrence"] for _, _, edge_data in graph.edges(node, data=True))
    for node in graph.nodes()
]

# Sử dụng thuật toán spring_layout để tự động phân bổ khoảng cách các node đẹp nhất
positions = nx.spring_layout(graph, k=0.55, seed=42)

# Độ dày của liên kết (cạnh) tỷ lệ thuận với số lần xuất hiện chung
edge_widths = [
    max(1.0, edge_data["co_occurrence"] / 20)
    for _, _, edge_data in graph.edges(data=True)
]

plt.figure(figsize=(15, 10))

nx.draw_networkx_edges(
    graph,
    positions,
    width=edge_widths,
    edge_color="#dddddd",
    alpha=0.7,
)

nx.draw_networkx_nodes(
    graph,
    positions,
    node_color=node_colors,
    node_size=node_sizes,
    edgecolors="none",
    alpha=0.95,
)

nx.draw_networkx_labels(
    graph,
    positions,
    font_size=8,
    font_family="sans-serif",
    font_weight="semibold",
)

# Tạo chú thích (legend) thủ công, trực quan bên góc đồ thị
plt.scatter([], [], color="#f768a1", s=150, label="Tiềm năng")
plt.scatter([], [], color="#7fcdbb", s=150, label="Chủ lực")
plt.legend(loc="upper left", scatterpoints=1, frameon=True, fontsize=10)

plt.title("Nhóm sản phẩm Tiềm Năng được mua kèm cùng các sản phẩm Chủ Lực", fontsize=15, fontweight="bold", pad=15)
plt.axis("off")
plt.tight_layout()
plt.show()

**Nhận xét:** Network graph cho thấy một số Hidden Gem lặp lại nhiều lần khi ghép với các Blockbuster. Ví dụ `Real2 Alkalized Water 500 ml` xuất hiện cùng nhiều sản phẩm chủ lực như `Bag of Organic Bananas`, `Organic Lemon`, `Organic Hass Avocado`, `Organic Baby Spinach`; cặp cao nhất trong lần chạy trên dataset này đạt **111 lần**. Điều này cho thấy Hidden Gem không nhất thiết có volume tổng lớn, nhưng khi đi cùng các sản phẩm phổ biến, chúng có thể trở thành ứng viên tốt cho recommendation hoặc bundle thử nghiệm.

## 8. Kết luận nhanh: Tổng quan về dataset

- Bộ dữ liệu Instacart mô tả hành vi mua sắm tạp hóa trực tuyến của khách hàng, bao gồm thông tin về đơn hàng, sản phẩm, nhóm hàng, thời điểm mua và trạng thái mua lại sản phẩm.

- Dataset có thể được phân tích ở nhiều cấp độ như đơn hàng, sản phẩm và khách hàng. Nhờ đó, nhóm có thể khám phá các xu hướng như thời điểm mua sắm phổ biến, kích thước giỏ hàng, sản phẩm bán chạy, sản phẩm có tỷ lệ mua lại cao và các cặp sản phẩm thường được mua cùng nhau.

- Tuy nhiên, dataset không có thông tin về giá bán, doanh thu hay lợi nhuận. Vì vậy, các phân tích về “giá trị khách hàng” chỉ nên được hiểu là chỉ báo hành vi, không phải giá trị tài chính thực tế.